In [894]:
import shutil
import networkx as nx
import dgl
import torch
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import random


In [895]:
import polygraphs as pg
from polygraphs import hyperparameters as hparams
from polygraphs import ops
from polygraphs.ops import math, complex
from polygraphs.analysis import Processor


# Generate Parameters

In [1033]:
def prejudice_params(size, groups, prejudiced_certainty, non_prejudiced_certainty, seed=None):
    """
    Create a PolyGraph configuration for a grouped complete network
    """
    params = hparams.PolyGraphHyperParameters()

    # Setting number of groups
    params.network.groups = groups

    # Setting prejudice and non-prejudice certainty for Jeffrey's Rule
    params.mistrust = prejudiced_certainty
    params.trust = non_prejudiced_certainty

    # Initial beliefs are random uniform between 0 and 1
    params.init.kind = 'uniform'
    # Chance that action B is better than action A
    params.epsilon = 0.5

    # drug_a = 0.5 effective rate

    # drug_b = drug_a + epsilon = 0.501 or 50.1%

    params.network.kind = 'complete_grouped'
    params.network.size = size
    params.network.selfloop = True

    params.simulation.steps = 1
    params.logging.enabled = False

    # Take snapshots (incl. messages)
    params.snapshots.enabled = True
    params.snapshots.interval = 100
    params.snapshots.messages = True


    # Set seed
    pg.random()

    if seed:
        # Explicitly set seed
        params.seed = seed
        pg.random(params.seed)

    return params

MAILBOX
group labels = [1,0,1,0]
payoffs = [[5,10], [6,10], [4,10], [5,10]] 


# Run Simulations

In [1023]:
groups_to_check = [2,3]
prejudiced_amounts_to_check = [0.5,0.75]
seeds_to_check = [i for i in range(51100, 51200)]
sizes_to_check = [16,64]

for size in sizes_to_check:
    for group in groups_to_check:
        for prejudiced_amount in prejudiced_amounts_to_check:
            for seed in seeds_to_check:
                params = prejudice_params(size=size, groups=group, prejudiced_certainty=prejudiced_amount, non_prejudiced_certainty=1, seed=seed)
                #(size, groups, prejudiced_certainty, non_prejudiced_certainty, seed)
                params.simulation.results = f"data3_control/EI-{size}-{group}-{prejudiced_amount}-{seed}"
                _ = pg.simulate(params, op=EpistemicInjusticeOp)
        

 INFO polygraphs> Sim #0001:   5302 steps   11.38s; action: B undefined: 0 converged: 1 polarized: 0 
 INFO polygraphs> Sim #0001:   4820 steps    9.26s; action: B undefined: 0 converged: 1 polarized: 0 
 INFO polygraphs> Sim #0001:   4723 steps    9.21s; action: B undefined: 0 converged: 1 polarized: 0 
 INFO polygraphs> Sim #0001:  11447 steps   22.27s; action: B undefined: 0 converged: 1 polarized: 0 
 INFO polygraphs> Sim #0001:   9806 steps   19.86s; action: B undefined: 0 converged: 1 polarized: 0 
 INFO polygraphs> Sim #0001:   9708 steps   18.83s; action: B undefined: 0 converged: 1 polarized: 0 
 INFO polygraphs> Sim #0001:  10819 steps   21.34s; action: B undefined: 0 converged: 1 polarized: 0 
 INFO polygraphs> Sim #0001:   8451 steps   17.37s; action: B undefined: 0 converged: 1 polarized: 0 
 INFO polygraphs> Sim #0001:   8986 steps   17.14s; action: B undefined: 0 converged: 1 polarized: 0 
 INFO polygraphs> Sim #0001:   8298 steps   16.00s; action: B undefined: 0 converg

# Custom Operation

In [1032]:
class EpistemicInjusticeOp(ops.core.PolyGraphOp):
    """
    Operation for simulating group-based distrust.
    Upon receipt, all nodes apply Jeffrey's rule.
    """

    def __init__(self, graph, params):
        super().__init__(graph, params)
        


        # Grabbing list of group labels
        self.groups = torch.unique(graph.ndata['group']).tolist()
        
        # Grabbing certainty value for Jeffrey's Rule where testimonial injustice occurs
        self.default_certainty = params.trust

        # Grabbing certainty values for Jeffrey's Rule where testimonial injustice does not occur
        self.prejudice_amount = params.mistrust

        #print("Total:",graph.ndata["group"])

        #  # Create group masks
        # self.group_masks = {}

        # for group in self.groups:
        #     self.group_masks[group] = (graph.ndata['group'] == group)
        # # print((graph.ndata['group']))
        # # print(self.group_masks[0])

    def messagefn(self):
        """
        Message function
        """

        #[a,b,c,d,e,f]
        # a, b
        # a: gives drug B to 10 patients, 5 recover
        # b: gives drug B to 10 patients, 6 recover
        # payoff a: [5,10]
        # payoff b: [6,10]

        # group labels: [0,1,0,1,2,2]
        # group label mask: mask = 2
        # [0,0,0,0,1,1]
        # group list: [0,1,2]

        def function(edges):
            return {
                "payoffs": edges.src["payoffs"],
                "group": edges.src["group"],
            }

        return function

    def reducefn(self):
        """
        Reduce function
        """

        def function(nodes):
            # Log probability of successful trials
            logits = nodes.data["logits"]
            # Prior, P(H) (aka. belief)
            prior = nodes.data["beliefs"]
            # Current Node's group
            recipient_group = nodes.data["group"]

            #print("Recipients:",recipient_group)
            ##print("Mailbox group:",nodes.mailbox["group"])
            ##print("All payoffs:",nodes.mailbox["payoffs"])

            # Duplicate each node's group label across columns for every neighbor
            # recipient_group_expanded = recipient_group.unsqueeze(1).expand(-1, nodes.mailbox["group"].shape[1])

            #print("expanded:",recipient_group_expanded)
            # print("Mailbox Groups")
            #print("mailbox group:",nodes.mailbox["group"])
            # print("mailbox payoffs shape:",nodes.mailbox["payoffs"].shape)
            #print("shape of nodes.data[group]",nodes.data["group"].shape)
            
            default_certainty = self.default_certainty
            prejudice_amount = self.prejudice_amount # Amount of discrediting for instances of testimonial injustice

            for group in self.groups:
                group_mask = (nodes.mailbox["group"] == group)
            
                masked_payoffs = nodes.mailbox["payoffs"] * group_mask.unsqueeze(-1).float()
                
                #print("Iterating over group:",group,)
                ##print("Masked Payoffs:",masked_payoffs)

                    # TOTAL CERTAINTY TENSOR, PROBABLY NOT NEEDED
                    # certainty_bin = (nodes.mailbox["group"] == 0) & (recipient_group_expanded == 1)
                    # #print("certainty",certainty_bin)

                    # # Define inverse mask
                    # certainty_bin_inv = ~certainty_bin
                    # #print("certainty inverse",certainty_bin_inv)

                    # # Set certainty in case of prejudice
                    # prejudiced_certainty = certainty_bin * 0.5 # Amount of discrediting, max: 0, min: 1

                    # # Set certainity for all other cases
                    # non_prejudiced_certainty = certainty_bin_inv * .9
                
                # Define nodes belonging to group 0 as prejudiced-against
                # (The certainty value for instances of testimonial injustice is applied only if the payoffs are sent by group 0)
                if group == 0:

                    # Define nodes belonging to group 1 as prejudiced
                    prejudiced_nodes_mask = (recipient_group == 1)

                    # Define all other nodes as non-prejudiced
                    non_prejudiced_nodes_mask = ~prejudiced_nodes_mask

                    # print(recipient_group)
                    # print("Prejudiced nodes:",prejudiced_nodes_mask)
                    # print(non_prejudiced_nodes_mask)

                    # Set certainty in case of prejudice
                    prejudiced_certainty = prejudiced_nodes_mask * prejudice_amount

                    # Set certainity for all other cases
                    non_prejudiced_certainty = non_prejudiced_nodes_mask * default_certainty

                    # Combine certainty tensors
                    certainty = prejudiced_certainty + non_prejudiced_certainty

                    #print("Combined certainty:",certainty)

                    # Sum values for all neighbors belonging to current sender group
                    sender_group_values = torch.sum(masked_payoffs[:, :, 0], dim=1)
                    ##print("Masked values sum:",sender_group_values)

                    # Sum trials for all neighbors belonging to current sender group
                    sender_group_trials = torch.sum(masked_payoffs[:, :, 1], dim=1)
                    #print("Masked trials sum:",sender_group_trials)

                    # Evidence, E
                    evidence = math.Evidence(logits, sender_group_values, sender_group_trials)
                    # print("Evidence:",evidence)

                    # Apply Jeffrey's Rule to all nodes
                    posterior = math.jeffrey(prior, evidence, certainty)

                    # Consider next sender group
                    prior = posterior


                else:

                    # Create a tensor with a 1 for each node in the network 
                    recipient_ones = torch.ones_like(recipient_group)
                    # Multiply 1's by default certainty
                    certainty = recipient_ones * default_certainty
                    #print("Combined certainty:",certainty)
                    
                    # Sum values for all neighbors belonging to current sender group
                    sender_group_values = torch.sum(masked_payoffs[:, :, 0], dim=1)
                    #print("masked values sum:",sender_group_values)

                    # Sum trials for all neighbors belonging to current sender group
                    sender_group_trials = torch.sum(masked_payoffs[:, :, 1], dim=1)
                    # print("masked trials sum:",sender_group_trials)

                    # Evidence, E
                    evidence = math.Evidence(logits, sender_group_values, sender_group_trials)
                    # print("Evidence:",evidence)

                    # Apply Jeffrey's Rule to all nodes
                    posterior = math.jeffrey(prior, evidence, certainty)
                    
                    # Consider next sender group
                    prior = posterior

            # Return posterior beliefs
            return {"beliefs": posterior}

        return function